# 12-bob. 12-BOB. XAVFSIZLIK OPERATSIYALARI MARKAZIDA SI
## Amaliy misol: signallarni saralash modeli

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/12-bob/12-2_signallarni-saralash-modeli.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 12-bob.*

Quyidagi dasturda xavfsizlik operatsiyalari markaziga 90 kun davomida kelgan 30 ming signal sun'iy ravishda yaratiladi. Signallar 40 ta turli qoidadan keladi. Har bir qoidaning muallif bergan jiddiylik darajasi (1 dan 4 gacha) bor, lekin u qoidaning haqiqiy sifati bilan bog'liq emas. Signalning haqiqiy hodisa bo'lish ehtimoli qoidaning sifatiga, aktivning muhimligiga, administrator hisobi ishtirok etganiga, tahdidlar bazasidagi mosliklarga, vaqtga va so'nggi 24 soatdagi o'xshash signallar soniga bog'liq (o'xshash signallar ko'p bo'lsa, bu odatda takrorlanuvchi noto'g'ri signal belgisidir). Har bir qoida uchun uning o'tmishdagi aniqligi faqat oldingi kunlardagi tahlilchi xulosalaridan hisoblanadi. Model dastlabki 60 kunda o'qitiladi va keyingi 30 kunda sinaladi. Uch xil tartib taqqoslanadi: kelish vaqti, qoida jiddiyligi va model bali bo'yicha. Dastur amaliy mashg'ulotning A12 mavzusiga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

rng = np.random.default_rng(11)
n = 30000                                                   # 90 kunlik ogohlantirishlar
qoidalar = pd.DataFrame({"qoida": range(40),
                         "jiddiylik": rng.integers(1, 5, 40),            # qoida muallifi bergan daraja (1-4)
                         "qoida_sifati": rng.beta(1, 6, 40)})            # qoidaning haqiqiy hodisa topish ehtimoli
df = pd.DataFrame({
    "kun": np.sort(rng.integers(0, 90, n)),
    "qoida": rng.integers(0, 40, n),
    "aktiv_muhimligi": rng.integers(1, 4, n),               # 1 - oddiy kompyuter, 3 - muhim server
    "admin_hisob": rng.random(n) < 0.1,
    "tahdid_razvedkasi": rng.random(n) < 0.03,              # IP yoki fayl tahdidlar bazasida bormi
    "tungi": rng.random(n) < 0.2,
    "oxshash_24s": rng.poisson(5, n),                       # so'nggi 24 soatdagi shunga o'xshash signallar
}).merge(qoidalar, on="qoida")

# Haqiqiy hodisa ehtimoli (tahlilchi xulosasi): bir nechta omilga bog'liq
logit = (-5.5 + 6 * df.qoida_sifati + 0.6 * df.aktiv_muhimligi + 1.0 * df.admin_hisob
         + 2.0 * df.tahdid_razvedkasi + 0.8 * df.tungi - 0.15 * df.oxshash_24s)
df["haqiqiy"] = rng.random(n) < 1 / (1 + np.exp(-logit))
df = df.sort_values("kun").reset_index(drop=True)
print(f"Ogohlantirishlar: {n}, ulardan haqiqiy hodisalar: {df.haqiqiy.mean():.1%}\n")

# Qoidaning o'tmishdagi aniqligi: faqat oldingi kunlardagi tahlilchi xulosalaridan
kunlik = df.groupby(["qoida", "kun"]).haqiqiy.agg(["sum", "count"]).groupby(level=0).cumsum()
kunlik = kunlik.groupby(level=0).shift().fillna(0)
df = df.join(((kunlik["sum"] + 1) / (kunlik["count"] + 10)).rename("qoida_tarixi"), on=["qoida", "kun"])

belgilar = ["jiddiylik", "aktiv_muhimligi", "admin_hisob", "tahdid_razvedkasi",
            "tungi", "oxshash_24s", "qoida_tarixi"]
oqitish, sinov = df[df.kun < 60], df[df.kun >= 60]
model = HistGradientBoostingClassifier(random_state=0).fit(oqitish[belgilar], oqitish.haqiqiy)
sinov = sinov.assign(ball=model.predict_proba(sinov[belgilar])[:, 1])

def kerakli_ulush(tartib, maqsad):
    """Haqiqiy hodisalarning `maqsad` qismini topish uchun signallarning qancha qismini ko'rish kerak."""
    topilgan = np.cumsum(sinov.loc[tartib, "haqiqiy"].values) / sinov.haqiqiy.sum()
    return (np.argmax(topilgan >= maqsad) + 1) / len(sinov)

tartiblar = {
    "Kelish vaqti bo'yicha": sinov.index,
    "Qoida jiddiyligi bo'yicha": sinov.sort_values("jiddiylik", ascending=False, kind="stable").index,
    "Model bali bo'yicha": sinov.sort_values("ball", ascending=False).index,
}
for nom, tartib in tartiblar.items():
    print(f"{nom:<26} hodisalarning 50% i: {kerakli_ulush(tartib, 0.5):5.1%}, "
          f"80% i: {kerakli_ulush(tartib, 0.8):5.1%} signalni ko'rish kerak")

Ogohlantirishlar: 30000, ulardan haqiqiy hodisalar: 3.4%



Kelish vaqti bo'yicha      hodisalarning 50% i: 52.1%, 80% i: 79.8% signalni ko'rish kerak
Qoida jiddiyligi bo'yicha  hodisalarning 50% i: 53.9%, 80% i: 84.4% signalni ko'rish kerak
Model bali bo'yicha        hodisalarning 50% i: 14.7%, 80% i: 42.1% signalni ko'rish kerak


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Ogohlantirishlar: 30000, ulardan haqiqiy hodisalar: 3.4%

Kelish vaqti bo'yicha      hodisalarning 50% i: 52.1%, 80% i: 79.8% signalni ko'rish kerak
Qoida jiddiyligi bo'yicha  hodisalarning 50% i: 53.9%, 80% i: 84.4% signalni ko'rish kerak
Model bali bo'yicha        hodisalarning 50% i: 14.7%, 80% i: 42.1% signalni ko'rish kerak
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 12-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.